## Metrics Explanation  

This section provides a detailed explanation of the metrics calculated in this notebook for analyzing movement patterns based on tracking data grouped by **patients (`ID`)** and sperm cells (`tracker_id`).  

### 1. **VCL (Curvilinear Velocity)**  
- **Definition**: VCL represents the average velocity of a tracker along its actual curvilinear path. It measures the distance traveled over time, calculated for each sperm within a patient's dataset.  
- **Mathematical Formula**:  

$$v_{ci} = \frac{\|p_{i+1} - p_i\| + \|p_i - p_{i-1}\|}{2\Delta t}$$
$$\text{VCL} = \frac{1}{N-2} \sum_{i=1}^{N-1} v_{ci}$$

  Where:  
  - $p_i$: Position at frame $i$ (e.g., $(x_{center}, y_{center})$).  
  - $\|p_{i+1} - p_i\|$: Euclidean distance between positions at frames \(i+1\) and \(i\).  
  - $\|p_i - p_{i-1}\|$: Euclidean distance between positions at frames \(i\) and \(i-1\).  
  - $\Delta t$: Time interval between frames.  
  - $N$: Total number of frames for the tracker.

### 2. **VSL (Straight-Line Velocity)**  
- **Definition**: VSL measures the average velocity along a straight line connecting the initial and final positions of a sperm cell. It quantifies the efficiency of movement in terms of directness.  
- **Mathematical Formula**:  
  $$\text{VSL} = \frac{\|p_{\text{end}} - p_{\text{start}}\|}{T}$$  
  Where:  
  - $p_{\text{start}}$: Starting position.  
  - $p_{\text{end}}$: Ending position.  
  - $T$: Total time ($N \cdot \Delta t$).  

### 3. **VAP (Average Path Velocity)**  
- **Definition**: VAP represents the average velocity along the path taken by the sperm cell, providing a measure of its average speed over time.  
- **Mathematical Formula**:  
  $$\text{total\_distance} = \sum_{i=1}^{n-1} \sqrt{(x_i - x_{i-1})^2 + (y_i - y_{i-1})^2}$$  
  $$\text{total\_time} = (n - 1) \times \Delta t$$  
  $$\text{VAP} = \frac{\text{total\_distance}}{\text{total\_time}}$$  
  Where:  
  - $x_i$, $y_i$: Coordinates of the position at frame $i$.  
  - $n$: Total number of frames for the tracker.  
  - $\Delta t$: Time interval between frames.  

### 4. **ALH (Amplitude of Lateral Head Displacement)**  
- **Definition**: ALH quantifies the average deviation of the tracker’s position from its average path. It reflects the lateral movement relative to the trajectory.  
- **Mathematical Formula**:  
  $$\text{ALH} = \frac{1}{N} \sum_{i=1}^{N} \|\bar{p} - p_i\|$$  
  Where:  
  - $\bar{p}$: Mean position of the tracker over all frames (average path center).  
  - $p_i$: Position at frame $i$.  

### 5. **MAD (Mean Angular Displacement)**  
- **Definition**: MAD measures the average angular displacement between three consecutive positions. It is useful for analyzing the directional changes in movement.  
- **Mathematical Formula**:  
  $$\theta_i = \cos^{-1}\left(\frac{(p_i - p_{i-1}) \cdot (p_{i+1} - p_i)}{\|p_i - p_{i-1}\| \cdot \|p_{i+1} - p_i\|}\right)$$  
  $$\text{MAD} = \frac{1}{N-2} \sum_{i=2}^{N-1} |\theta_i|$$  
  Where:  
  - $\theta_i$: Angle between vectors formed by three consecutive points.  
  - $\cdot$: Dot product of two vectors.  
  - $\| \cdot \|$: Magnitude of a vector.  

---

### Notes  
- Metrics are computed for each tracker ID grouped by patient ID.  
- $N$: Total number of frames per sperm tracker.  
- All calculations assume the data is ordered temporally (e.g., by `ID`).  
- Metrics such as VCL, VSL, and VAP are expressed in units of distance per time (e.g., $\mu m/s$), while ALH is expressed in units of distance (e.g., $\mu m$), and MAD is in degrees.

# Imports

In [51]:
import pandas as pd
from nb_utils import set_root
import numpy as np
import sys
import json
import os
from pathlib import Path
from typing import List, Union


PROJECT_DIR = set_root(2)

# Parameters

In [52]:
path_data = PROJECT_DIR / "data"
path_intermediate = path_data / "02_intermediate"
path_primary = path_data / "03_primary"

file_path_data = path_primary / "tracker_cut.parquet"
file_path_horm = path_intermediate / "data_horm_concat.parquet"
tracker_columns = ["tracker_id",	"class_id",	"x_min",	"y_min",	"x_max",	"y_max",	"x_center",	"y_center"]

# Data

In [53]:
data = pd.read_parquet(file_path_data)
data.head()

,tracker_id,class_id,x_min,y_min,x_max,y_max,ID,x_center,y_center
0,0,0,81.517593,341.390228,98.150589,358.730377,6,89.834091,350.060303
1,1,0,220.177231,32.698105,235.510880,48.191841,6,227.844055,40.444973
2,2,0,501.025208,244.382629,518.151794,260.651978,6,509.588501,252.517303
3,3,0,168.232040,412.327515,191.797409,435.265625,6,180.014725,423.796570
4,4,0,445.648376,83.616653,466.867371,104.786850,6,456.257874,94.201752


In [54]:
data[data['ID']=='11']

,tracker_id,class_id,x_min,y_min,x_max,y_max,ID,x_center,y_center
82442,0,0,11.996006,340.508911,27.073059,358.137268,11,19.534533,349.323090
82443,1,0,117.560654,397.132751,131.427689,411.739441,11,124.494171,404.436096
82444,2,0,144.475739,270.416077,160.486328,287.952332,11,152.481033,279.184204
82445,3,0,169.494431,190.922363,183.491104,205.117096,11,176.492767,198.019730
82446,4,0,447.627930,300.581818,460.620239,314.848846,11,454.124084,307.715332
...,...,...,...,...,...,...,...,...,...
100823,287,0,161.899887,199.085022,177.744797,216.246338,11,169.822342,207.665680
100824,76,0,42.218666,379.363281,60.855659,398.134033,11,51.537163,388.748657
100825,56,0,356.720398,43.213913,374.168823,61.779602,11,365.444611,52.496758
100826,341,0,4.883986,344.664032,22.569681,364.998810,11,13.726833,354.831421


# Functions

In [62]:
def set_root(level: int = 1) -> Path:
    for i in range(level):
        if i == 0:
            PROJECT_DIR = Path.cwd().parent
        else:
            PROJECT_DIR = PROJECT_DIR.parent
    sys.path.append(str(PROJECT_DIR))
    return PROJECT_DIR

def find_name_with_prefix(names: List[str], prefix: str) -> Union[None|str]:
    for name in names:
        if name.startswith(prefix):
            return name
    return None

def generate_path_url(content: Union[pd.DataFrame|np.ndarray], path_video: Union[Path|str], target_col: Union[str|int]= "ID"):
    path_url = {}
    for idx in content[target_col].unique():
        files_path = os.listdir(path_video)
        file_name = find_name_with_prefix(files_path, str(idx) + "_")
        if file_name:
            path_url[idx] = str(path_video / file_name)
    return path_url



def position_values(group):
    x = group['x_center'].iloc[-1]
    y = group['y_center'].iloc[-1]
    
    return pd.Series({'x': x, 'y': y})

def calculate_metrics(group):
    positions = list(zip(group['x_center'], group['y_center']))
    delta_t = 0.02
    
    vcl = calculate_vcl(positions, delta_t)
    vsl = calculate_vsl(positions)
    vap = calculate_vap(positions, delta_t)
    alh = calculate_alh(positions)
    mad = calculate_mad(positions)
    last_positions = position_values(group)
    return pd.Series({'x': last_positions['x'], 'y': last_positions['y'] ,'VCL': vcl, 'VSL': vsl, 'VAP': vap, "ALH": alh, "MAD": mad})

def create_group_id(df, x):
    df['time'] = df.groupby('tracker_id').cumcount() // x
    #df['group_id'] = df['group_id'] + 1
    return df



#Funções Novas
def calculate_vcl(temp, delta_t=0.02):
    if len(temp) < 3:
        return 0
    a = []
    for idx in range(1, len(temp) - 1):
        x_center, y_center = temp[idx]
        x_center_minus, y_center_minus = temp[idx - 1]
        x_center_plus, y_center_plus = temp[idx + 1]
        norm_minus = np.linalg.norm(np.array([x_center, y_center]) - np.array([x_center_minus, y_center_minus]))
        norm_plus = np.linalg.norm(np.array([x_center_plus, y_center_plus]) - np.array([x_center, y_center]))
        numerador = norm_minus + norm_plus
        denominador = 2 * delta_t
        vci = numerador / denominador
        a.append(vci)
    return sum(a) / (len(temp) - 2)

def calculate_vsl(temp):
    if len(temp) == 1:
        return 0
    init_value = temp[0]
    final_value = temp[-1]
    vsl = np.sqrt(((init_value[0] - final_value[0])**2) + ((init_value[1] - final_value[1]) ** 2))/(len(temp)*0.02)
    return vsl

def calculate_vap(temp, delta_t=0.02):
    if len(temp) < 3:
        return 0
    a = []
    for idx in range(1, len(temp) - 1):
        x_center, y_center = temp[idx]
        x_center_minus, y_center_minus = temp[idx-1]
        a.append(np.sqrt(((x_center - x_center_minus)**(2)) + ((y_center - y_center_minus)**(2))))
    total_time = (len(temp) - 1) * delta_t
    return sum(a) / total_time
def calculate_alh(temp):
    mean_position = np.mean(temp, axis=0)
    a = []
    for idx in range(len(temp)):
        a.append(np.linalg.norm(mean_position - temp[idx]))
    alh = sum(a) / len(a)
    return alh

def calculate_mad(temp):
    if len(temp) < 3:
        return 0
    angles = []
    for idx in range(1, len(temp) - 1):
        p_i_minus_1 = np.array(temp[idx - 1])
        p_i = np.array(temp[idx])
        p_i_plus_1 = np.array(temp[idx + 1])

        vector_1 = p_i - p_i_minus_1
        vector_2 = p_i_plus_1 - p_i
        
        dot_product = np.dot(vector_1, vector_2)
        magnitude_1 = np.linalg.norm(vector_1)
        magnitude_2 = np.linalg.norm(vector_2)
        
        if magnitude_1 == 0 or magnitude_2 == 0:
            continue
        
        cos_theta = dot_product / (magnitude_1 * magnitude_2)
        theta = np.arccos(np.clip(cos_theta, -1.0, 1.0))
        angles.append(np.abs(theta))
    
    if len(angles) == 0:
        return 0
    
    mad = sum(angles) / len(angles)
    return mad


#Funções que geram os json
def metrics_dataframe_to_json(metrics_df, path_url_trackeado, path_url_nao_trackeado):
    data = {'metrics': []}
    for patient_id, patient_group in metrics_df.groupby('ID'):
        patient_metrics = {'id': int(patient_id), 'trackers': []}
        for _, row in patient_group.iterrows():
            tracker_metrics = {
                'tracker_id': int(row['tracker_id']),
                'VCL': round(row['VCL'], 2),
                'VSL': round(row['VSL'], 2),
                'VAP': round(row['VAP'], 2),
                'ALH': round(row['ALH'], 2),
                'MAD': round(row['MAD'], 2),
            }
            patient_metrics['trackers'].append(tracker_metrics)
        patient_metrics["video_url_trackeado"] = path_url_trackeado.get(patient_id)
        patient_metrics["video_url_nao_trackeado"] = path_url_nao_trackeado.get(patient_id)
        data['metrics'].append(patient_metrics)
    return json.dumps(data, indent=4)


def save_json_to_directory(json_data, filename):
    """
    Saves the JSON data to a specific directory 'visualization/outputs' located 
    one level up from the current working directory.

    Args:
        json_data (str): JSON formatted string.
        filename (str): Name of the JSON file.
    """
    parent_directory = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
    target_directory = os.path.join(parent_directory, 'visualization', 'outputs')
    
    if not os.path.exists(target_directory):
        os.makedirs(target_directory)
    
    file_path = os.path.join(target_directory, filename)
    

    with open(file_path, 'w') as f:
        f.write(json_data)
    
    print(f"JSON saved to {file_path}")



def dataframe_to_json(df, grouped):
    """
    Converts the dataframe to the specified JSON format.

    Args:
        df (pd.DataFrame): DataFrame with metrics and positions.
        grouped (pd.DataFrame): Grouped DataFrame with additional route information.

    Returns:
        str: JSON formatted string.
    """
    data = {'individuos': []}
    grouped_dict = {}
    for _, row in grouped.iterrows():
        key = (row['ID'], row['tracker_id'])
        if key not in grouped_dict:
            grouped_dict[key] = []
        grouped_dict[key].append({'x': row['x_center'], 'y': row['y_center']})

    for patient_id, patient_group in df.groupby('ID'):
        individual = {'id': int(patient_id), 'espermatozoides': []}

        for tracker_id, group in patient_group.groupby('tracker_id'):
            espermatozoide = {'id': int(tracker_id), 'route': [], 'frames': []}
            key = (patient_id, tracker_id)
            if key in grouped_dict:
                espermatozoide['route'].extend(grouped_dict[key])

            for _, row in group.iterrows():
                espermatozoide['frames'].append({
                    'x': round(row['x'], 2), #ok
                    'y': round(row['y'], 2), #ok
                    'VCL': round(row['VCL'], 2),
                    'VSL': round(row['VSL'], 2),
                    'VAP': round(row['VAP'], 2),
                    'ALH': round(row['ALH'], 2),
                    'MAD': round(row['MAD'], 2)
                })

            individual['espermatozoides'].append(espermatozoide)
        data['individuos'].append(individual)

    return json.dumps(data, indent=4)


# Calculate metrics

In [64]:
data

,tracker_id,class_id,x_min,y_min,x_max,y_max,ID,x_center,y_center
0,0,0,81.517593,341.390228,98.150589,358.730377,6,89.834091,350.060303
1,1,0,220.177231,32.698105,235.510880,48.191841,6,227.844055,40.444973
2,2,0,501.025208,244.382629,518.151794,260.651978,6,509.588501,252.517303
3,3,0,168.232040,412.327515,191.797409,435.265625,6,180.014725,423.796570
4,4,0,445.648376,83.616653,466.867371,104.786850,6,456.257874,94.201752
...,...,...,...,...,...,...,...,...,...
2196284,2091,0,523.772156,161.258850,539.751770,181.233521,69,531.761963,171.246185
2196285,1829,0,461.298218,186.087952,476.646240,202.813721,69,468.972229,194.450836
2196286,3239,0,94.208496,424.543518,107.303009,438.854553,69,100.755753,431.699036
2196287,3254,0,435.511108,461.199219,446.554077,477.938171,69,441.032593,469.568695


In [65]:
#Calculo por janelas de tempo
fps = 50 #temos que as imagens foram captadas a uma velocidade de 50 frames por segundo
#grouped = data.sort_values(['ID', 'tracker_id']).drop('class_id', axis=1).reset_index()
#cria grupos de acordo com o fps que a gente definiu
grouped = data.groupby(['ID', 'tracker_id']).apply(create_group_id, x=fps).reset_index(drop=True)

#Calcula métricas de janela
result = grouped.groupby(['ID', 'tracker_id', 'time']).apply(calculate_metrics).reset_index()
result.head()

C:\Users\ccana\AppData\Local\Temp\ipykernel_10564\857757640.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped = data.groupby(['ID', 'tracker_id']).apply(create_group_id, x=fps).reset_index(drop=True)
C:\Users\ccana\AppData\Local\Temp\ipykernel_10564\857757640.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  result = grouped.groupby(['ID', 'tracker_id', 'time']).apply(calculate_metrics).reset_in

,ID,tracker_id,time,x,y,VCL,VSL,VAP,ALH,MAD
0,1,0,0,152.466644,335.840515,0.0,0.0,0.0,0.0,0.0
1,1,1,0,492.019775,245.750488,0.0,0.0,0.0,0.0,0.0
2,1,2,0,270.667603,157.405945,0.0,0.0,0.0,0.0,0.0
3,1,3,0,340.559387,429.905579,0.0,0.0,0.0,0.0,0.0
4,1,4,0,635.276001,471.097137,0.0,0.0,0.0,0.0,0.0


In [66]:
#Calculo de métricas gerais
metrics_df = data.groupby(['ID', 'tracker_id']).apply(calculate_metrics).reset_index()
metrics_df


C:\Users\ccana\AppData\Local\Temp\ipykernel_10564\27621040.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  metrics_df = data.groupby(['ID', 'tracker_id']).apply(calculate_metrics).reset_index()


,ID,tracker_id,x,y,VCL,VSL,VAP,ALH,MAD
0,1,0,152.466644,335.840515,0.0,0.000000,0.0,0.000000,0.0
1,1,1,492.019775,245.750488,0.0,0.000000,0.0,0.000000,0.0
2,1,2,270.667603,157.405945,0.0,0.000000,0.0,0.000000,0.0
3,1,3,340.559387,429.905579,0.0,0.000000,0.0,0.000000,0.0
4,1,4,635.276001,471.097137,0.0,0.000000,0.0,0.000000,0.0
...,...,...,...,...,...,...,...,...,...
89295,9,664,412.340393,229.448547,0.0,22.433698,0.0,0.448674,0.0
89296,9,665,4.720243,371.604004,0.0,10.396068,0.0,0.207921,0.0
89297,9,666,25.478252,203.962021,0.0,4.721611,0.0,0.094432,0.0
89298,9,667,569.693909,246.499084,0.0,0.000000,0.0,0.000000,0.0


In [68]:
temp = metrics_df[metrics_df["ID"] == "47"].copy()
temp[temp["tracker_id"] == 19]

,ID,tracker_id,x,y,VCL,VSL,VAP,ALH,MAD
36619,47,19,299.707947,60.25959,105.400111,49.913625,104.921995,75.864033,0.757528


In [69]:
temp = data[data["ID"] == "47"].copy()
temp = temp[temp["tracker_id"] == 19]

In [61]:
np.linalg.norm(temp[["x_center", "y_center"]].values[0] - temp[["x_center", "y_center"]].values[-1])

245.71518

# Data to JSON

In [70]:
# URLs dos vídeos trackeados
path_video_trackeado = PROJECT_DIR / "data" / "03_primary" / "tracker_video"
path_url_trackeado = generate_path_url(metrics_df, path_video_trackeado)

# URLs dos vídeos não trackeados
path_video_nao_trackeado = PROJECT_DIR / "datasets" / "data" / "visem" / "visem-dataset" / "video_cut"
path_url_nao_trackeado = generate_path_url(metrics_df, path_video_nao_trackeado)

json_dt = metrics_dataframe_to_json(metrics_df, path_url_trackeado, path_url_nao_trackeado)
save_json_to_directory(json_dt, 'metrics_general.json')

#json de acordo com a janela de tempo que foi definida no início
json_data = dataframe_to_json(result, grouped)
save_json_to_directory(json_data, 'data_window.json')


JSON saved to c:\Users\ccana\Documents\Doutorado\VIS1170\cin-dataviz\visualization\outputs\metrics_general.json
JSON saved to c:\Users\ccana\Documents\Doutorado\VIS1170\cin-dataviz\visualization\outputs\data_window.json
